# Phase 3 — Which external knowledge helps the 5-image tail? (research study)

**Research question:** *for the 5-image tail, which kind of external knowledge helps most —*
*language (LLM-enriched prototypes), a second vision foundation model (DINOv2), or generative*
*data (diffusion in feature space) — and do they complement each other?*

We measure each knowledge source against the **from-scratch control** (`cmo`, the best model
trained on our data alone), debias the foundation models fairly with **GLA**, and test
complementarity by **fusing** experts with tail-aware (per-shot-group) weights. Everything
runs on **cached** frozen features, so it fits a Kaggle session.

Builds on: `clip_expert`/`lift` (Phase 2), `vlm_fusion`/`tier_fusion` (Phase 0),
`balanced_softmax` (GLA generalises it), and the `cmo` checkpoint (Phase 1).

## 0. Setup (open_clip + transformers; DINOv2 via torch.hub)

In [ ]:
import importlib.util, sys, subprocess
for pkg, mod in [("open_clip_torch", "open_clip"), ("transformers", "transformers")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("deps ready")

## 1. Make `src/` importable

In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if base.exists():
            candidates += [p for p in base.iterdir() if p.is_dir()]
    for path in candidates:
        if (path / "src" / "__init__.py").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_root()
sys.path.insert(0, str(PROJECT_DIR))
print("Project root:", PROJECT_DIR)

## 2. Config

Toggle each knowledge source. `VAL_FRACTION`/`SEED` match the other notebooks.
`CMO_DIR` must hold the Phase-1 `best_model.pt` (the from-scratch control).

In [ ]:
DATA_DIR = PROJECT_DIR / "data" / "CIFAR-100-LT"
OUTPUT_DIR = PROJECT_DIR / "outputs"
NUM_CLASSES = 100
BATCH_SIZE = 128
NUM_WORKERS = 2
DEVICE = None
SEED = 42
VAL_FRACTION = 0.1

# Shot-group thresholds (train images/class). CIFAR-100-LT: 100 / 20.
# CUB-200-LT (max ~30/class): use 15 / 6.
MANY_THRESHOLD = 100
FEW_THRESHOLD = 20

CLIP_MODEL = "ViT-B-32"
CLIP_PRETRAINED = "openai"

# --- which knowledge sources to run ---
USE_LLM = True          # A: LLM-enriched CLIP text prototypes
USE_DINO = True         # D: DINOv2 second vision backbone
USE_DIFFUSION = True    # C: diffusion feature augmentation for the tail
USE_GLA = True          # B: debias the foundation models (ablation)
USE_CMO = True          # from-scratch control expert (needs the checkpoint)
USE_MIXUP = True        # tail-aware feature mixup (CMO idea in feature space)
CMO_DIR = OUTPUT_DIR / "cmo"

# --- module knobs ---
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
N_DESC_PER_CLASS = 8
DESC_JSON = OUTPUT_DIR / "class_descriptions.json"   # cache: LLM runs once
DINO_MODEL = "dinov2_vits14"
LIFT_EPOCHS = 50
LIFT_LR = 1e-3
DIFFUSION_STEPS = 100
DIFFUSION_EPOCHS = 200
MIXUP_ALPHA = 1.0       # Beta(alpha, alpha) mixing strength

MAX_TRAIN_SAMPLES = None   # smoke test: e.g. 2000

## 3. Imports, device, data, feature cache

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_class_names, load_split, make_loader,
                                    split_indices_by_class, split_shot_groups, subset)
from src.evaluation import visualize
from src.evaluation.metrics import compute_metrics, format_metrics
from src.evaluation import fusion
from src.experts.clip_expert import (clip_zero_shot_logits, encode_clip_features,
                                     encode_text_prototypes, load_clip)
from src.experts import gla, lift as lift_mod
from src.utils.experiment import compare_runs, create_run_dir, save_metrics
from src.utils.seed import resolve_device, set_seed

set_seed(SEED)
device = resolve_device(DEVICE)
class_counts = load_class_counts(DATA_DIR)
NUM_CLASSES = len(class_counts)            # auto-match the dataset (100 CIFAR-100, 200 CUB)
CLASS_NAMES = load_class_names(DATA_DIR)   # readable names for CLIP / LLM prompts
shot_groups = split_shot_groups(class_counts, MANY_THRESHOLD, FEW_THRESHOLD)

eval_tf = build_transforms(train=False, image_size=32)
base_train = load_split(DATA_DIR, "train", eval_tf)
train_idx, val_idx = split_indices_by_class([l for _, l in base_train.samples], VAL_FRACTION, SEED)
train_eval, val_eval = subset(base_train, train_idx), subset(base_train, val_idx)
test_eval = load_split(DATA_DIR, "test", eval_tf)

clip_bundle = load_clip(CLASS_NAMES, device, model_name=CLIP_MODEL, pretrained=CLIP_PRETRAINED)
f_train, y_train = encode_clip_features(train_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
f_val, y_val = encode_clip_features(val_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
f_test, y_test = encode_clip_features(test_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(y_train):
    keep = torch.randperm(len(y_train), generator=torch.Generator().manual_seed(SEED))[:MAX_TRAIN_SAMPLES]
    f_train, y_train = f_train[keep], y_train[keep]
train_counts = list(np.bincount(y_train.numpy(), minlength=NUM_CLASSES))
yv, yt = y_val.numpy(), y_test.numpy()

# Expert registry: name -> per-sample logits/probs on val and test (aligned to yv / yt).
EXP_VAL, EXP_TEST = {}, {}

def add_expert(name, val_scores, test_scores):
    EXP_VAL[name] = np.asarray(val_scores)
    EXP_TEST[name] = np.asarray(test_scores)
    record(name, EXP_TEST[name].argmax(1))

def record(name, y_pred):
    m = compute_metrics(yt, y_pred, NUM_CLASSES, shot_groups)
    save_metrics(create_run_dir(OUTPUT_DIR, name), m)
    print(f"{name:18s} bal={m['balanced_accuracy']:.4f}  many={m['many_shot_accuracy']:.3f}  "
          f"med={m['medium_shot_accuracy']:.3f}  few={m['few_shot_accuracy']:.3f}")
    return m

print("features:", f_train.shape, "| device:", device)

## 4. Baseline expert — CLIP zero-shot (plain prompts)

In [ ]:
clip_val = clip_zero_shot_logits(f_val, clip_bundle).numpy()
clip_test = clip_zero_shot_logits(f_test, clip_bundle).numpy()
add_expert("clip_zeroshot", clip_val, clip_test)
proto_init = clip_bundle["text_features"].float()   # LIFT cosine-head init (may be replaced by A)

## 5. Module A — LLM-enriched class prototypes (language knowledge)

Generate rich per-class descriptions with a free LLM (cached to JSON), average their
CLIP text embeddings into better prototypes, and re-run zero-shot. These prototypes also
seed LIFT's head in section 6.

In [ ]:
if USE_LLM:
    from src.experts import llm_prompts
    if DESC_JSON.exists():
        descriptions = llm_prompts.load_descriptions(DESC_JSON)
        print("loaded cached descriptions:", len(descriptions), "classes")
    else:
        print("generating descriptions with", LLM_MODEL, "(runs once)...")
        descriptions = llm_prompts.generate_descriptions(CLASS_NAMES, device,
                                                         llm_model_name=LLM_MODEL, n_per_class=N_DESC_PER_CLASS)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        llm_prompts.save_descriptions(descriptions, DESC_JSON)
    print("example [leopard]:", descriptions.get("leopard", ["-"])[0][:90])

    prompts = llm_prompts.prompts_in_label_order(descriptions, CLASS_NAMES)
    proto_llm = encode_text_prototypes(clip_bundle, prompts, device)
    add_expert("clip_llm", (clip_bundle["logit_scale"] * f_val @ proto_llm.cpu().t()).numpy(),
                           (clip_bundle["logit_scale"] * f_test @ proto_llm.cpu().t()).numpy())
    proto_init = proto_llm.float()   # use the richer prototypes to seed LIFT

## 6. CLIP + LIFT (strong vision-language expert)

In [ ]:
def lift_expert(feats_tr, lab_tr, init, name, **kw):
    model, info = lift_mod.train_lift(feats_tr, lab_tr, f_val, y_val, init.to(device),
                                      clip_bundle["logit_scale"], list(np.bincount(lab_tr.numpy(), minlength=NUM_CLASSES)),
                                      NUM_CLASSES, device, epochs=LIFT_EPOCHS, lr=LIFT_LR, **kw)
    with torch.no_grad():
        vl = model(f_val.to(device)).cpu().numpy()
        tl = model(f_test.to(device)).cpu().numpy()
    print(f"  [{name}] best val bal={info['best_val_balanced_accuracy']:.4f}")
    return vl, tl

vl, tl = lift_expert(f_train, y_train, proto_init, "lift_clip")
add_expert("lift_clip", vl, tl)

## 7. Module C — diffusion feature augmentation (generative knowledge)

Train a small class-conditional DDPM on CLIP features, synthesise extra **tail** features
to balance the set, then train LIFT on real + synthetic. Tests whether *manufactured* tail
data helps beyond what we have.

In [ ]:
if USE_DIFFUSION:
    from src.experts import feature_diffusion as fd
    diff = fd.FeatureDiffusion(f_train.shape[1], NUM_CLASSES, device, num_steps=DIFFUSION_STEPS)
    diff.train(f_train, y_train, epochs=DIFFUSION_EPOCHS)
    f_aug, y_aug = fd.augment_tail(f_train, y_train, diff, NUM_CLASSES)
    print(f"  augmented {len(y_train)} -> {len(y_aug)} features")
    vl, tl = lift_expert(f_aug, y_aug, proto_init, "lift_clip_diff")
    add_expert("lift_clip_diff", vl, tl)

## 7b. Tail-aware feature mixup (the CMO idea, in feature space)

Mix each feature with a class-balanced (tail-rich) partner and soft-mix the labels — the
feature-space analogue of CMO (Method 6). A second *augmentation* knowledge source to compare
against diffusion (`lift_clip_diff`): does mixing real features or synthesising new ones help
the tail more? `balanced_softmax` is already the training loss inside every LIFT expert.

In [ ]:
if USE_MIXUP:
    vl, tl = lift_expert(f_train, y_train, proto_init, "lift_clip_mixup", mixup_alpha=MIXUP_ALPHA)
    add_expert("lift_clip_mixup", vl, tl)

## 8. Module D — DINOv2 (second vision foundation model)

Self-supervised, no language. Cache features once, train LIFT with class-mean (NCM) init.

In [ ]:
if USE_DINO:
    from src.experts import dino_expert
    dino_bundle = dino_expert.load_dino(device, model_name=DINO_MODEL)
    d_train, _ = dino_expert.encode_dino_features(train_eval, dino_bundle, device, BATCH_SIZE, NUM_WORKERS)
    d_val, _ = dino_expert.encode_dino_features(val_eval, dino_bundle, device, BATCH_SIZE, NUM_WORKERS)
    d_test, _ = dino_expert.encode_dino_features(test_eval, dino_bundle, device, BATCH_SIZE, NUM_WORKERS)
    if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(d_train):
        d_train = d_train[keep]
    proto_dino = dino_expert.class_mean_prototypes(d_train, y_train, NUM_CLASSES)
    dm, dinfo = lift_mod.train_lift(d_train, y_train, d_val, y_val, proto_dino.to(device), 30.0,
                                    train_counts, NUM_CLASSES, device, epochs=LIFT_EPOCHS, lr=LIFT_LR)
    with torch.no_grad():
        add_expert("dino_lift", dm(d_val.to(device)).cpu().numpy(), dm(d_test.to(device)).cpu().numpy())

### 8b. Baseline: DINOv2 + plain linear probe (isolates our contribution)

A vanilla `Linear` head trained with **CrossEntropy** (no adapter, no cosine head, no
class-mean init, no Balanced Softmax) on the *same* frozen DINOv2 features, same val-selection.
The gap `dino_lift - dino_linear_probe` is exactly what **our LIFT pipeline** adds on top of the
raw backbone (expected: mostly a `few_shot` gain from Balanced Softmax). Recorded for the table
only — **not** fed into the fusion (it would be redundant with `dino_lift`).

In [ ]:
if USE_DINO:
    import copy
    import torch.nn as nn
    from src.evaluation.metrics import balanced_accuracy

    def linear_probe(ftr, ytr, fva, yva, fte, num_classes, device, epochs, lr=1e-3, criterion=None):
        head = nn.Linear(ftr.shape[1], num_classes).to(device)
        opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        crit = criterion if criterion is not None else nn.CrossEntropyLoss()  # default: plain CE (raw-backbone baseline)
        loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(ftr, ytr.long()),
                                             batch_size=256, shuffle=True)
        best_state, best = copy.deepcopy(head.state_dict()), -1.0
        for _ in range(epochs):
            head.train()
            for xb, yb in loader:
                loss = crit(head(xb.to(device)), yb.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()
            head.eval()
            with torch.no_grad():
                v = balanced_accuracy(yva.numpy(), head(fva.to(device)).argmax(1).cpu().numpy())
            if v > best:
                best, best_state = v, copy.deepcopy(head.state_dict())
        head.load_state_dict(best_state); head.eval()
        with torch.no_grad():
            return head(fte.to(device)).cpu().numpy()

    tl_lp = linear_probe(d_train, y_train, d_val, y_val, d_test, NUM_CLASSES, device, LIFT_EPOCHS)
    record("dino_linear_probe", tl_lp.argmax(1))   # table only; not added to fusion

### 8c. DINOv2 ablation — add Balanced Softmax (isolate loss vs architecture)

Same linear head, but trained with **Balanced Softmax** instead of CE. This splits our LIFT
contribution into two attributable steps:

`dino_linear_probe` (Linear+CE)  ->  `dino_linear_probe_bs` (Linear+**BalancedSoftmax**)  ->  `dino_lift` (+adapter+cosine+class-mean init)

- gap (bs − ce) = the **tail-aware loss** alone (expect a large `few_shot` jump),
- gap (lift − bs) = the **adapter + cosine head + class-mean init** on top.

In [ ]:
if USE_DINO:
    from src.trainers.losses import BalancedSoftmaxLoss
    tl_bs = linear_probe(d_train, y_train, d_val, y_val, d_test, NUM_CLASSES, device,
                         LIFT_EPOCHS, criterion=BalancedSoftmaxLoss(train_counts).to(device))
    record("dino_linear_probe_bs", tl_bs.argmax(1))   # table only; not added to fusion

## 9. From-scratch control — `cmo` (no external knowledge)

The Phase-1 vision model trained on our data alone: the mock everything else must beat
to justify *borrowing* knowledge.

In [ ]:
if USE_CMO and (CMO_DIR / "best_model.pt").exists():
    from src.evaluation.ensemble import load_classifier, predict_probs
    cmo = load_classifier(CMO_DIR, NUM_CLASSES, device, pretrained=False)
    pv, yv_c = predict_probs(cmo, make_loader(val_eval, BATCH_SIZE, False, NUM_WORKERS), device)
    pt, yt_c = predict_probs(cmo, make_loader(test_eval, BATCH_SIZE, False, NUM_WORKERS), device)
    assert (yv_c == yv).all() and (yt_c == yt).all(), "cmo sample order mismatch"
    add_expert("cmo", pv, pt)          # stored as probabilities; fusion.to_probs handles it
else:
    print("cmo checkpoint not found -> skipping control (set CMO_DIR / run run_all_methods)")

## 10. Module B — GLA debiasing (fair comparison)

Remove each foundation expert's own pretraining label bias (estimated transductively on
the test logits, strength tuned on val). Reported as an ablation; `strength=0` is in the
grid so it never hurts.

In [ ]:
if USE_GLA:
    import torch as _t
    print("GLA debiasing (before -> after, balanced acc):")
    for name in [n for n in ("clip_zeroshot", "clip_llm", "lift_clip", "dino_lift") if n in EXP_TEST]:
        vlog, tlog = _t.tensor(EXP_VAL[name]), _t.tensor(EXP_TEST[name])
        bias = gla.estimate_log_bias(tlog)
        s, _ = gla.tune_strength(vlog, yv, gla.estimate_log_bias(vlog))
        before = compute_metrics(yt, tlog.argmax(1).numpy(), NUM_CLASSES, shot_groups)["balanced_accuracy"]
        after = compute_metrics(yt, gla.gla_adjust(tlog, bias, s).argmax(1).numpy(), NUM_CLASSES, shot_groups)["balanced_accuracy"]
        print(f"  {name:16s} {before:.4f} -> {after:.4f}  (strength={s:.2f})")
        # keep the debiased scores for fusion
        EXP_VAL[name] = gla.gla_adjust(vlog, gla.estimate_log_bias(vlog), s).numpy()
        EXP_TEST[name] = gla.gla_adjust(tlog, bias, s).numpy()

## 11. Complementarity — tail-aware fusion of the knowledge sources

Tune per-shot-group weights on val, fuse, and compare every single expert vs the fusion
**per shot-group**. If the fusion beats the best single expert (especially on `few`), the
knowledge sources complement each other — the answer to the research question.

In [ ]:
val_probs = {k: fusion.to_probs(v) for k, v in EXP_VAL.items()}
test_probs = {k: fusion.to_probs(v) for k, v in EXP_TEST.items()}
names = list(test_probs.keys())
print("experts in fusion:", names)

group_w, val_bal = fusion.tune_group_weights([val_probs[n] for n in names], yv, shot_groups, NUM_CLASSES)
class_group = np.empty(NUM_CLASSES, dtype=object)
for g, ids in shot_groups.items():
    for c in ids:
        class_group[c] = g
fused_test = fusion.fuse_group([test_probs[n] for n in names], group_w, class_group)
add_expert("fusion_tailaware", fused_test, fused_test)   # logits==probs here; argmax identical

print("\nper-shot-group weights (over", names, "):")
for g in ("many", "medium", "few"):
    print(f"  {g:7s}", tuple(round(w, 2) for w in group_w[g]))

rep = fusion.complementarity_report(test_probs, yt, NUM_CLASSES, shot_groups, fused_test)
rep_df = pd.DataFrame(rep).T[["balanced_accuracy", "many_shot_accuracy", "medium_shot_accuracy", "few_shot_accuracy"]]
rep_df = rep_df.sort_values("balanced_accuracy", ascending=False)
rep_df.to_csv(OUTPUT_DIR / "knowledge_sources.csv")
rep_df.round(4)

### 11b. Greedy ensemble fusion (no per-group regression, by construction)

Added *alongside* the weighted fusion above (nothing modified). Uses **greedy ensemble
selection** (Caruana et al., ICML 2004; the modern "greedy soup", Wortsman et al. 2022):
per shot-group, start from the **best single expert** and add an expert only if it improves
that group's balanced val accuracy. By construction the result is **>= best single expert on
val** for every present group — so it does not regress on `medium` the way the weighted grid
can when one expert (DINOv2) dominates. The `few` group inherits `medium` (val has no tail).

In [ ]:
# Greedy fusion: monotone-improving over the best single per group (Caruana 2004).
gw_greedy, valbal_greedy = fusion.greedy_group_weights(
    [val_probs[n] for n in names], yv, shot_groups, NUM_CLASSES)
fused_greedy = fusion.fuse_group([test_probs[n] for n in names], gw_greedy, class_group)
record("fusion_greedy", fused_greedy.argmax(1))   # table only; not re-fed into any fusion
print(f"greedy val bal={valbal_greedy:.4f} | per-group weights (over {names}):")
for g in ("many", "medium", "few"):
    print(f"  {g:7s}", tuple(round(w, 2) for w in gw_greedy[g]))

## 12. Answering the research question

Read off the table above:
- **Which knowledge helps the tail most?** compare `few_shot_accuracy` across `clip_llm`
  (language), `dino_lift` (second vision), `lift_clip_diff` (generative).
- **Do they complement?** does `fusion_tailaware` beat the best single expert (esp. on `few`)?
- **Is external knowledge needed?** compare everything to `cmo` (from-scratch control).
- **Did debiasing (GLA) help?** see section 10's before/after.

In [ ]:
comparison = compare_runs(OUTPUT_DIR)
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
vlm = ["clip_zeroshot", "clip_llm", "lift_clip", "lift_clip_diff", "lift_clip_mixup", "dino_lift", "dino_linear_probe", "dino_linear_probe_bs", "fusion_tailaware", "fusion_greedy", "cmo"]
comparison[comparison["method"].isin(vlm)].reset_index(drop=True)